In [1]:
import numpy as np
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# ----------------------------------------
# Load data
# ----------------------------------------

"""
Bespoke generative decoders for trial-averaged Bernoulli features

Implements (A) Beta Naive Bayes on averaged probabilities x_ij in (0,1)
and (B) Beta-Binomial Naive Bayes if you know trial count T (uses k_ij ≈ round(T*x_ij)).

Outputs:
- evidence matrix Phi (same shape as X): phi_ij = log p(x_ij|y=1) - log p(x_ij|y=0)
- generative score s_i = logit(pi) + sum_j phi_ij
- optional PLS reduction on Phi + logistic regression in reduced space

Designed for large p (e.g. 40k neurons). Uses chunking and vectorization.
"""

import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression

from scipy.special import betaln, gammaln


# ============================================================
# NUMERICS
# ============================================================

def _clip01(x, eps=1e-6):
    return np.clip(x, eps, 1.0 - eps)

def _logit(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def _beta_logpdf(x, a, b):
    # x: (n, p), a,b: (p,)
    # returns (n, p)
    return (a - 1.0) * np.log(x) + (b - 1.0) * np.log(1.0 - x) - betaln(a, b)

def _betabinom_logpmf(k, T, a, b):
    # k: (n, p) ints, a,b: (p,)
    # log [ C(T,k) * B(k+a, T-k+b) / B(a,b) ]
    # returns (n,p)
    k = k.astype(np.int64)
    logC = gammaln(T + 1) - gammaln(k + 1) - gammaln(T - k + 1)
    return logC + betaln(k + a, (T - k) + b) - betaln(a, b)


# ============================================================
# ESTIMATION: mu_jc and shared kappa
# ============================================================

def estimate_mu_per_class(X, y, a0=0.5, b0=0.5):
    """
    X: (n, p) in (0,1)
    y: (n,) in {0,1}
    Returns mu0, mu1: (p,)
    Uses a Beta(a0,b0) pseudo-count shrinkage on the mean:
      mu = (sum x + a0) / (n_c + a0 + b0)
    """
    X = _clip01(X)
    y = y.astype(int)
    idx0 = (y == 0)
    idx1 = (y == 1)
    n0 = max(int(idx0.sum()), 1)
    n1 = max(int(idx1.sum()), 1)

    s0 = X[idx0].sum(axis=0) if idx0.any() else np.zeros(X.shape[1])
    s1 = X[idx1].sum(axis=0) if idx1.any() else np.zeros(X.shape[1])

    mu0 = (s0 + a0) / (n0 + a0 + b0)
    mu1 = (s1 + a0) / (n1 + a0 + b0)
    return _clip01(mu0), _clip01(mu1)

def estimate_shared_kappa(X, y, mu0, mu1, kappa_min=2.0, kappa_max=1e6):
    """
    Shared concentration kappa using a pooled method-of-moments:
      Var[ X | class c ] ≈ mu_c(1-mu_c) / (kappa + 1)
    We estimate average empirical variance across neurons and classes,
    and solve for kappa.
    """
    X = _clip01(X)
    y = y.astype(int)
    idx0 = (y == 0)
    idx1 = (y == 1)

    # Empirical variances per neuron within each class (ddof=1 if possible)
    def safe_var(A):
        if A.shape[0] <= 1:
            return np.zeros(A.shape[1])
        return A.var(axis=0, ddof=1)

    v0 = safe_var(X[idx0]) if idx0.any() else np.zeros(X.shape[1])
    v1 = safe_var(X[idx1]) if idx1.any() else np.zeros(X.shape[1])

    # Expected beta variance numerator per class
    num0 = mu0 * (1.0 - mu0)
    num1 = mu1 * (1.0 - mu1)

    # Pool across classes and neurons robustly
    num = 0.5 * (num0 + num1)
    den = 0.5 * (v0 + v1)

    # Avoid dividing by ~0 variances (super-stable neurons)
    mask = den > np.percentile(den, 10)  # keep the more informative 90%
    if mask.sum() < 10:
        mask = den > 0

    if mask.sum() == 0:
        return 50.0  # fallback

    num_m = float(np.mean(num[mask]))
    den_m = float(np.mean(den[mask]))

    # kappa ≈ num/var - 1
    kappa = (num_m / max(den_m, 1e-12)) - 1.0
    kappa = float(np.clip(kappa, kappa_min, kappa_max))
    return kappa


# ============================================================
# GENERATIVE DECODERS
# ============================================================

class BetaNaiveBayesEvidence:
    """
    x_ij | y=c ~ Beta(alpha_jc, beta_jc)
    alpha_jc = mu_jc * kappa, beta_jc = (1-mu_jc)*kappa
    Shared kappa across neurons and classes (estimated from data unless provided).
    """

    def __init__(self, a0=0.5, b0=0.5, kappa=None, eps=1e-6, chunk=4096):
        self.a0 = a0
        self.b0 = b0
        self.kappa = kappa
        self.eps = eps
        self.chunk = chunk

        # fitted
        self.mu0_ = None
        self.mu1_ = None
        self.alpha0_ = None
        self.beta0_ = None
        self.alpha1_ = None
        self.beta1_ = None
        self.pi_ = None

    def fit(self, X, y):
        X = _clip01(np.asarray(X, float), self.eps)
        y = np.asarray(y, int)

        self.pi_ = float(np.mean(y))
        mu0, mu1 = estimate_mu_per_class(X, y, self.a0, self.b0)
        self.mu0_, self.mu1_ = mu0, mu1

        kappa = self.kappa
        if kappa is None:
            kappa = estimate_shared_kappa(X, y, mu0, mu1)
        self.kappa = float(kappa)

        self.alpha0_ = _clip01(mu0, self.eps) * self.kappa
        self.beta0_  = (1.0 - _clip01(mu0, self.eps)) * self.kappa
        self.alpha1_ = _clip01(mu1, self.eps) * self.kappa
        self.beta1_  = (1.0 - _clip01(mu1, self.eps)) * self.kappa
        return self

    def evidence_matrix(self, X):
        """
        Phi_ij = log p(x_ij|y=1) - log p(x_ij|y=0)
        Returns Phi of shape (n, p). Chunked over neurons for memory.
        """
        X = _clip01(np.asarray(X, float), self.eps)
        n, p = X.shape
        Phi = np.empty((n, p), dtype=np.float32)

        for j0 in range(0, p, self.chunk):
            j1 = min(p, j0 + self.chunk)
            Xc = X[:, j0:j1]
            ll1 = _beta_logpdf(Xc, self.alpha1_[j0:j1], self.beta1_[j0:j1])
            ll0 = _beta_logpdf(Xc, self.alpha0_[j0:j1], self.beta0_[j0:j1])
            Phi[:, j0:j1] = (ll1 - ll0).astype(np.float32)
        return Phi

    def score(self, X):
        """
        s_i = logit(pi) + sum_j phi_ij
        """
        Phi = self.evidence_matrix(X)
        return _logit(self.pi_) + Phi.sum(axis=1)

    def predict_proba(self, X):
        s = self.score(X)
        p1 = 1.0 / (1.0 + np.exp(-s))
        return np.vstack([1 - p1, p1]).T

    def predict(self, X, thresh=0.5):
        return (self.predict_proba(X)[:, 1] >= thresh).astype(int)


class BetaBinomialNaiveBayesEvidence:
    """
    If you know T (trials per image), use k_ij | y=c ~ BetaBinomial(T, alpha_jc, beta_jc)
    We take k_ij ≈ round(T * x_ij) if only x_ij is available.
    Same mu/kappa parameterization for stability.
    """

    def __init__(self, T, a0=0.5, b0=0.5, kappa=None, eps=1e-6, chunk=4096):
        self.T = int(T)
        self.a0 = a0
        self.b0 = b0
        self.kappa = kappa
        self.eps = eps
        self.chunk = chunk

        self.mu0_ = None
        self.mu1_ = None
        self.alpha0_ = None
        self.beta0_ = None
        self.alpha1_ = None
        self.beta1_ = None
        self.pi_ = None

    def fit(self, X, y):
        X = _clip01(np.asarray(X, float), self.eps)
        y = np.asarray(y, int)

        self.pi_ = float(np.mean(y))
        mu0, mu1 = estimate_mu_per_class(X, y, self.a0, self.b0)
        self.mu0_, self.mu1_ = mu0, mu1

        kappa = self.kappa
        if kappa is None:
            kappa = estimate_shared_kappa(X, y, mu0, mu1)
        self.kappa = float(kappa)

        self.alpha0_ = _clip01(mu0, self.eps) * self.kappa
        self.beta0_  = (1.0 - _clip01(mu0, self.eps)) * self.kappa
        self.alpha1_ = _clip01(mu1, self.eps) * self.kappa
        self.beta1_  = (1.0 - _clip01(mu1, self.eps)) * self.kappa
        return self

    def evidence_matrix(self, X):
        X = _clip01(np.asarray(X, float), self.eps)
        n, p = X.shape
        Phi = np.empty((n, p), dtype=np.float32)

        k = np.rint(self.T * X).astype(np.int64)  # approximate counts
        k = np.clip(k, 0, self.T)

        for j0 in range(0, p, self.chunk):
            j1 = min(p, j0 + self.chunk)
            kc = k[:, j0:j1]
            ll1 = _betabinom_logpmf(kc, self.T, self.alpha1_[j0:j1], self.beta1_[j0:j1])
            ll0 = _betabinom_logpmf(kc, self.T, self.alpha0_[j0:j1], self.beta0_[j0:j1])
            Phi[:, j0:j1] = (ll1 - ll0).astype(np.float32)
        return Phi

    def score(self, X):
        Phi = self.evidence_matrix(X)
        return _logit(self.pi_) + Phi.sum(axis=1)

    def predict_proba(self, X):
        s = self.score(X)
        from scipy.special import expit
        p1 = expit(s)
        return np.vstack([1 - p1, p1]).T

    def predict(self, X, thresh=0.5):
        return (self.predict_proba(X)[:, 1] >= thresh).astype(int)


# ============================================================
# PLS ON EVIDENCE + LOGISTIC
# ============================================================

def fit_pls_logistic(Phi_train, y_train, Phi_test, n_components=5, C=1.0, max_iter=2000):
    """
    PLSRegression learns supervised components of Phi for y.
    Then logistic regression is fit on the PLS scores.

    Returns: (yhat_test, proba_test, pls, clf, Z_train, Z_test)
    """
    # PLS wants y as float column
    pls = PLSRegression(n_components=n_components, scale=False)
    pls.fit(Phi_train, y_train.astype(float).reshape(-1, 1))

    Z_train = pls.transform(Phi_train)
    Z_test  = pls.transform(Phi_test)

    clf = LogisticRegression(penalty="l2", C=C, solver="lbfgs", max_iter=max_iter)
    clf.fit(Z_train, y_train)

    proba = clf.predict_proba(Z_test)[:, 1]
    yhat = (proba >= 0.5).astype(int)
    return yhat, proba, pls, clf, Z_train, Z_test


# ============================================================
# DEMO: LOAD YOUR DATA + RUN TRUE vs PERM CV
# ============================================================


VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'

vit = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']
R   = np.load(NEURAL_PATH).T

top1 = np.argmax(vit, axis=1)
y = (top1 <= 397).astype(int)
X = np.clip(R.astype(float), 1e-6, 1 - 1e-6)

n = X.shape[0]
loo = LeaveOneOut()

# ----------------------------------------
# Storage for error patterns
# columns: [BetaNB, Logistic, SVM-RBF]
# ----------------------------------------
E = np.zeros((n, 3), dtype=int)

# ----------------------------------------
# LOO loop
# ----------------------------------------
for i, (tr, te) in enumerate(loo.split(X)):
    Xtr, Xte = X[tr], X[te]
    ytr, yte = y[tr], y[te]

    # ========= 1. Your Beta-Binomial NB =========
    m = BetaBinomialNaiveBayesEvidence(T=50, a0=0.5, b0=0.5)
    m.fit(Xtr, ytr)
    yhat_gen = m.predict(Xte)[0]
    E[i, 0] = int(yhat_gen != yte[0])

    # ========= 2. Logistic Regression =========
    scaler = StandardScaler(with_mean=False)
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)

    clf_lr = LogisticRegression(penalty="l2", C=1.0, max_iter=5000)
    clf_lr.fit(Xtr_s, ytr)
    yhat_lr = clf_lr.predict(Xte_s)[0]
    E[i, 1] = int(yhat_lr != yte[0])

    # ========= 3. Nonlinear SVM (RBF) =========
    clf_svm = SVC(kernel="rbf", C=1.0, gamma="scale")
    clf_svm.fit(Xtr_s, ytr)
    yhat_svm = clf_svm.predict(Xte_s)[0]
    E[i, 2] = int(yhat_svm != yte[0])

    if (i+1) % 10 == 0:
        print(f"LOO {i+1}/{n} done")

# ----------------------------------------
# Error covariance matrix
# ----------------------------------------
Cov = np.cov(E, rowvar=False)

print("\nError matrix shape:", E.shape)
print("\nError covariance matrix:")
print(Cov)

np.save("/home/maria/ProjectionSort/data/error_matrix.npy", E)
np.save("/home/maria/ProjectionSort/data/error_covariance.npy", Cov)


LOO 10/118 done
LOO 20/118 done
LOO 30/118 done
LOO 40/118 done
LOO 50/118 done
LOO 60/118 done
LOO 70/118 done
LOO 80/118 done
LOO 90/118 done
LOO 100/118 done
LOO 110/118 done

Error matrix shape: (118, 3)

Error covariance matrix:
[[0.22316384 0.14624076 0.11205273]
 [0.14624076 0.22316384 0.13769376]
 [0.11205273 0.13769376 0.22316384]]


In [3]:
np.corrcoef(E.T)

array([[1.        , 0.65530672, 0.5021097 ],
       [0.65530672, 1.        , 0.61700747],
       [0.5021097 , 0.61700747, 1.        ]])

In [6]:
import numpy as np

E = np.load("/home/maria/ProjectionSort/data/error_matrix.npy")

names = ["BetaNB", "Logistic", "SVM"]

def pairwise_agreement(E, names):
    n_models = E.shape[1]
    n = E.shape[0]

    for i in range(n_models):
        for j in range(i+1, n_models):
            ei = E[:, i]
            ej = E[:, j]

            both_wrong = np.sum((ei == 1) & (ej == 1))
            both_right = np.sum((ei == 0) & (ej == 0))
            i_wrong_j_right = np.sum((ei == 1) & (ej == 0))
            i_right_j_wrong = np.sum((ei == 0) & (ej == 1))

            print(f"\n{names[i]} vs {names[j]}")
            print("--------------------------------")
            print(f"Both wrong      : {both_wrong}")
            print(f"Both correct    : {both_right}")
            print(f"{names[i]} wrong, {names[j]} correct : {i_wrong_j_right}")
            print(f"{names[i]} correct, {names[j]} wrong : {i_right_j_wrong}")
            print(f"Agreement rate  : {(both_wrong + both_right)/n:.3f}")

def jaccard_failures(E, names):
    n_models = E.shape[1]

    for i in range(n_models):
        for j in range(i+1, n_models):
            ei = E[:, i]
            ej = E[:, j]

            inter = np.sum((ei == 1) & (ej == 1))
            union = np.sum((ei == 1) | (ej == 1))
            jacc = inter / union if union > 0 else np.nan

            print(f"Jaccard({names[i]}, {names[j]}) = {jacc:.3f}")

pairwise_agreement(E,names)
jaccard_failures(E,names)


BetaNB vs Logistic
--------------------------------
Both wrong      : 30
Both correct    : 70
BetaNB wrong, Logistic correct : 9
BetaNB correct, Logistic wrong : 9
Agreement rate  : 0.847

BetaNB vs SVM
--------------------------------
Both wrong      : 26
Both correct    : 66
BetaNB wrong, SVM correct : 13
BetaNB correct, SVM wrong : 13
Agreement rate  : 0.780

Logistic vs SVM
--------------------------------
Both wrong      : 29
Both correct    : 69
Logistic wrong, SVM correct : 10
Logistic correct, SVM wrong : 10
Agreement rate  : 0.831
Jaccard(BetaNB, Logistic) = 0.625
Jaccard(BetaNB, SVM) = 0.500
Jaccard(Logistic, SVM) = 0.592


In [7]:
import numpy as np

ERR_PATH = "/home/maria/ProjectionSort/data/error_matrix.npy"

names = ["BetaNB", "Logistic", "SVM"]

E = np.load(ERR_PATH)
n_models = E.shape[1]
n = E.shape[0]

print(f"Total images: {n}\n")

for i in range(n_models):
    for j in range(i+1, n_models):
        ei = E[:, i]
        ej = E[:, j]

        both_wrong   = np.sum((ei == 1) & (ej == 1))
        both_right   = np.sum((ei == 0) & (ej == 0))
        discordant   = np.sum(ei != ej)
        total_agree  = both_wrong + both_right

        print(f"{names[i]} vs {names[j]}")
        print("-" * 40)
        print(f"Both wrong (agreeing errors) : {both_wrong}")
        print(f"Both right                  : {both_right}")
        print(f"Discordant results          : {discordant}")
        print(f"Total agreement             : {total_agree}")
        print(f"Agreement rate              : {total_agree / n:.3f}\n")


Total images: 118

BetaNB vs Logistic
----------------------------------------
Both wrong (agreeing errors) : 30
Both right                  : 70
Discordant results          : 18
Total agreement             : 100
Agreement rate              : 0.847

BetaNB vs SVM
----------------------------------------
Both wrong (agreeing errors) : 26
Both right                  : 66
Discordant results          : 26
Total agreement             : 92
Agreement rate              : 0.780

Logistic vs SVM
----------------------------------------
Both wrong (agreeing errors) : 29
Both right                  : 69
Discordant results          : 20
Total agreement             : 98
Agreement rate              : 0.831



In [8]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

ERR_PATH = "/home/maria/ProjectionSort/data/error_matrix.npy"
OUT_PATH = "/home/maria/ProjectionSort/data/error_pca.png"

E = np.load(ERR_PATH)          # shape (118, 3)
n = E.shape[0]

# --------------------------------------
# PCA
# --------------------------------------
pca = PCA(n_components=2)
Z = pca.fit_transform(E)

print("Explained variance ratio:", pca.explained_variance_ratio_)

# Number of models that failed per image
fail_count = E.sum(axis=1)

# --------------------------------------
# Plot
# --------------------------------------
plt.figure(figsize=(6, 5))
sc = plt.scatter(Z[:,0], Z[:,1], c=fail_count, s=60)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of decoder error patterns")

cbar = plt.colorbar(sc)
cbar.set_label("Number of models wrong")

for i in range(n):
    plt.text(Z[i,0]+0.01, Z[i,1]+0.01, str(i), fontsize=7, alpha=0.6)

plt.tight_layout()
plt.savefig(OUT_PATH, dpi=200)
plt.close()

print("Saved PCA plot to:", OUT_PATH)


Explained variance ratio: [0.72841606 0.16675672]
Saved PCA plot to: /home/maria/ProjectionSort/data/error_pca.png


In [9]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

ERR_PATH = "/home/maria/ProjectionSort/data/error_matrix.npy"

names = ["BetaNB", "Logistic", "SVM"]

E = np.load(ERR_PATH)        # (118, 3)
M = E.T                     # (3, 118)

# -----------------------------------
# PCA in model space
# -----------------------------------
pca = PCA(n_components=2)
Z = pca.fit_transform(M)

print("Explained variance:", pca.explained_variance_ratio_)

# -----------------------------------
# Plot
# -----------------------------------
plt.figure(figsize=(5, 5))
plt.scatter(Z[:,0], Z[:,1], s=120)

for i, name in enumerate(names):
    plt.text(Z[i,0]+0.02, Z[i,1]+0.02, name, fontsize=11)

plt.axhline(0, ls="--", alpha=0.3)
plt.axvline(0, ls="--", alpha=0.3)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Geometry of decoder error fingerprints")
plt.tight_layout()
plt.savefig("/home/maria/ProjectionSort/data/model_error_geometry.png", dpi=200)
plt.close()


Explained variance: [0.61267348 0.38732652]
